# Near-Half Court Detector — Training Sweep (v2)

**v2 changes:** MSE loss (was BCE), full-res 640×360 training (was 80×45), fixed LR scheduler feedback loop, relabeled 283 frames via CVAT, recalibrated `lambda_vis`.

**Setup:**
1. Upload `tennis_vision_training.zip` to the **root** of your Google Drive (`My Drive/`).
2. Set the runtime to **T4 GPU**: Runtime → Change runtime type → T4 GPU.
3. **Run All** (Runtime → Run all).

This notebook runs a diagnostic calibration for `lambda_vis`, then a learning-rate sweep over 3 backbone LR values:
- `1e-5`, `3e-5`, `1e-4`

Results and best checkpoints are copied back to Google Drive when finished.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract training zip to /content and verify layout
import os, zipfile

ZIP_PATH = '/content/drive/MyDrive/tennis_vision_training.zip'
EXTRACT_DIR = '/content'

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

os.chdir('/content')
print(f'Working directory: {os.getcwd()}')

# Verify critical files/dirs exist
expected = [
    'src',
    'near_half_train/labels.json',
    'near_half_train/crops',
    'court_detector.pt',
    'train_near_half_court.py',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ MISSING'
    print(f'  {status}  {p}')

!ls /content

In [ ]:
# Install dependencies
!pip install -q albumentations opencv-python-headless tqdm pyyaml

In [ ]:
# Verify GPU, imports, and data
import torch

assert torch.cuda.is_available(), 'No GPU detected — change runtime to T4 GPU!'
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f'GPU: {props.name}  |  VRAM: {vram_gb:.1f} GB')

# Verify dataset imports work
from src.features.court_detect.dataset import create_train_val_datasets, TEST_VIDEO_STEMS, VAL_VIDEO_STEMS
print(f'Val stems: {VAL_VIDEO_STEMS}')
print(f'Test stems: {TEST_VIDEO_STEMS}')

# Quick data sanity check
import json
with open('near_half_train/labels.json') as f:
    labels = json.load(f)
print(f'Total labelled frames: {len(labels)}')

# Verify a sample image path resolves correctly
sample = labels[0]
img_rel = sample['image_path']  # e.g. 'crops/xxx.jpg'
img_full = os.path.join('near_half_train', img_rel)
assert os.path.isfile(img_full), f'Sample image not found: {img_full}'
print(f'Sample image OK: {img_full}')

# Create datasets to confirm everything wires up
train_ds, val_ds = create_train_val_datasets(
    labels_json='near_half_train/labels.json',
    images_dir='near_half_train',
)
print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}')

In [ ]:
# Create output directories
!mkdir -p logs weights/diagnostic weights/sweep_lr1e5 weights/sweep_lr3e5 weights/sweep_lr1e4

In [ ]:
%%time
# Diagnostic: calibrate lambda_vis (runs 3 epochs, ~27 min on T4)
# The loss function changed from BCE to MSE, so we need to recalibrate
!python train_near_half_court.py \
    --data-dir near_half_train \
    --labels-json near_half_train/labels.json \
    --pretrained-weights court_detector.pt \
    --diagnostic \
    --num-workers 2 \
    --output-dir weights/diagnostic

**⚠️ Read the diagnostic output above.** Copy the recommended `lambda_vis` value and paste it into the sweep cells below before running them. The old value (0.11) was calibrated with BCE loss and is no longer valid.

In [ ]:
%%time
# Sweep 1/3 — backbone-lr = 1e-5
!python train_near_half_court.py \
    --data-dir near_half_train \
    --labels-json near_half_train/labels.json \
    --pretrained-weights court_detector.pt \
    --backbone-lr 1e-5 \
    --lambda-vis 0.XX \  # ← UPDATE from diagnostic output above
    --num-workers 2 \
    --output-dir weights/sweep_lr1e5

In [ ]:
%%time
# Sweep 2/3 — backbone-lr = 3e-5
!python train_near_half_court.py \
    --data-dir near_half_train \
    --labels-json near_half_train/labels.json \
    --pretrained-weights court_detector.pt \
    --backbone-lr 3e-5 \
    --lambda-vis 0.XX \  # ← UPDATE from diagnostic output above
    --num-workers 2 \
    --output-dir weights/sweep_lr3e5

In [ ]:
%%time
# Sweep 3/3 — backbone-lr = 1e-4
!python train_near_half_court.py \
    --data-dir near_half_train \
    --labels-json near_half_train/labels.json \
    --pretrained-weights court_detector.pt \
    --backbone-lr 1e-4 \
    --lambda-vis 0.XX \  # ← UPDATE from diagnostic output above
    --num-workers 2 \
    --output-dir weights/sweep_lr1e4

In [ ]:
# Compare results across sweep runs
import torch, glob, os

sweep_configs = [
    ('1e-5', 'weights/sweep_lr1e5'),
    ('3e-5', 'weights/sweep_lr3e5'),
    ('1e-4', 'weights/sweep_lr1e4'),
]

rows = []
for lr_label, out_dir in sweep_configs:
    # Find the best checkpoint file
    candidates = sorted(glob.glob(os.path.join(out_dir, '*best*')))
    if not candidates:
        candidates = sorted(glob.glob(os.path.join(out_dir, '*.pt')))
    if not candidates:
        rows.append({'lr': lr_label, 'file': 'NO CHECKPOINT FOUND'})
        continue

    ckpt_path = candidates[0]
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    row = {'lr': lr_label, 'file': os.path.basename(ckpt_path)}
    # Extract metrics if stored in checkpoint
    for key in ('val_loss', 'best_val_loss', 'epoch', 'val_kp_error', 'val_vis_acc'):
        if isinstance(ckpt, dict) and key in ckpt:
            val = ckpt[key]
            row[key] = f'{val:.4f}' if isinstance(val, float) else val
    rows.append(row)

# Print comparison table
if rows:
    all_keys = list(dict.fromkeys(k for r in rows for k in r.keys()))
    header = ' | '.join(f'{k:>14s}' for k in all_keys)
    print(header)
    print('-' * len(header))
    for r in rows:
        vals = [str(r.get(k, '—')) for k in all_keys]
        print(' | '.join(f'{v:>14s}' for v in vals))

In [ ]:
# Copy results back to Google Drive
import shutil

DRIVE_OUT = '/content/drive/MyDrive/tennis_vision_sweep_results'
os.makedirs(DRIVE_OUT, exist_ok=True)

for lr_label, out_dir in sweep_configs:
    dest = os.path.join(DRIVE_OUT, os.path.basename(out_dir))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(out_dir, dest)
    print(f'Copied {out_dir} → {dest}')

# Also copy diagnostic results
if os.path.isdir('weights/diagnostic') and os.listdir('weights/diagnostic'):
    diag_dest = os.path.join(DRIVE_OUT, 'diagnostic')
    if os.path.exists(diag_dest):
        shutil.rmtree(diag_dest)
    shutil.copytree('weights/diagnostic', diag_dest)
    print(f'Copied weights/diagnostic → {diag_dest}')

# Also copy logs if any
if os.path.isdir('logs') and os.listdir('logs'):
    logs_dest = os.path.join(DRIVE_OUT, 'logs')
    if os.path.exists(logs_dest):
        shutil.rmtree(logs_dest)
    shutil.copytree('logs', logs_dest)
    print(f'Copied logs → {logs_dest}')

print(f'\n✅ All results saved to: {DRIVE_OUT}')